## product meta

In [ ]:
import pandas as pd
from pathlib import Path
import yaml
import os

product_meta_static_path = Path(r'D:\量化策略\CTA策略\Multi-factor Trend Following\db\rules_ingestion\data\derived\20260624\product_meta_static\yaml')
product_meta_price_tick_path = Path(r'D:\量化策略\CTA策略\Multi-factor Trend Following\db\rules_ingestion\data\derived\20260624\product_meta_price_tick_rule\yaml')
product_meta_session_path = Path(r'D:\量化策略\CTA策略\Multi-factor Trend Following\db\rules_ingestion\data\derived\20260624\product_meta_trading_sessions\V2\yaml')

# find all yaml file in product_meta_static_path
all_yaml_paths = product_meta_static_path.glob('*.yaml')


for path in all_yaml_paths:
    # extract the name of the file
    product = path.stem
    print(f'Combining product meta yaml for {product}')
    
    session_yaml_path = product_meta_session_path / f'{product}.yaml'
    price_tick_yaml_path = product_meta_price_tick_path / f'{product}.yaml'

    product_meta_dict = yaml.safe_load(path.read_text(encoding='utf-8'))
    product_session_dict = yaml.safe_load(session_yaml_path.read_text(encoding='utf-8'))
    product_price_tick_dict = yaml.safe_load(price_tick_yaml_path.read_text(encoding='utf-8'))

    product_meta_str = yaml.safe_dump(product_meta_dict, default_flow_style=False, sort_keys=False, allow_unicode=True)
    product_session_str = yaml.safe_dump(product_session_dict, default_flow_style=False, sort_keys=False, allow_unicode=True)
    product_price_tick_str = yaml.safe_dump(product_price_tick_dict, default_flow_style=False, sort_keys=False, allow_unicode=True)

    combined_yaml_str = (
        '# -------------------- static Meta --------------------\n' + product_meta_str + '\n' + 
        '# -------------------- price tick rule --------------------\n' + product_price_tick_str + '\n' +
        '# -------------------- sessions history -------------------\n' + product_session_str + '\n'
    )
    out_path = Path(r'D:\量化策略\CTA策略\Multi-factor Trend Following\db\rules\product_meta') / f'{product}.yaml'
    out_path.write_text(combined_yaml_str, encoding='utf-8')
    

    



Combining product meta yaml for A
Combining product meta yaml for AD
Combining product meta yaml for AG
Combining product meta yaml for AL
Combining product meta yaml for AO
Combining product meta yaml for AP
Combining product meta yaml for AU
Combining product meta yaml for B
Combining product meta yaml for BB
Combining product meta yaml for BC
Combining product meta yaml for BR
Combining product meta yaml for BU
Combining product meta yaml for BZ
Combining product meta yaml for C
Combining product meta yaml for CF
Combining product meta yaml for CJ
Combining product meta yaml for CS
Combining product meta yaml for CU
Combining product meta yaml for CY
Combining product meta yaml for EB
Combining product meta yaml for EC
Combining product meta yaml for EG
Combining product meta yaml for ER
Combining product meta yaml for FB
Combining product meta yaml for FG
Combining product meta yaml for FU
Combining product meta yaml for HC
Combining product meta yaml for I
Combining product meta y

## Daily Status

In [ ]:
daily_status_dir = Path(r'D:\量化策略\CTA策略\Multi-factor Trend Following\db\rules_ingestion\data\derived\20260624\instrument_daily_status')
daily_status_order_limits_dir = Path(r'D:\量化策略\CTA策略\Multi-factor Trend Following\db\rules_ingestion\data\derived\20260624\instrument_daily_status_order_volume_limits')

all_parquet_paths = daily_status_dir.glob('*.parquet')

missing_order_limits_products = []
for path in all_parquet_paths:
    print(f'Combining daily status df, exchange_product: {path.stem}')
    df = pd.read_parquet(path)
    df = df.set_index(['instrument', 'trading_date'])
    order_limits_file_path = daily_status_order_limits_dir / path.name

    if not order_limits_file_path.exists():
        missing_order_limits_products.append(path.stem)
        continue

    df_order_limits = pd.read_parquet(order_limits_file_path)
    df_order_limits = df_order_limits.set_index(['instrument', 'trading_date'])
    # concat two dfs
    combined_daily_status_df = pd.concat([df, df_order_limits]).reset_index()
    
    out_path = Path(r'D:\量化策略\CTA策略\Multi-factor Trend Following\db\rules\instrument_daily_status') / path.name
    combined_daily_status_df.to_parquet(out_path, index=False)

print(f'所有缺失 order limits parquet 文件的品种为: {missing_order_limits_products}')

Combining daily status df, exchange_product: CFFEX_IC
Combining daily status df, exchange_product: CFFEX_IF
Combining daily status df, exchange_product: CFFEX_IH
Combining daily status df, exchange_product: CFFEX_IM
Combining daily status df, exchange_product: CFFEX_T
Combining daily status df, exchange_product: CFFEX_TF
Combining daily status df, exchange_product: CFFEX_TL
Combining daily status df, exchange_product: CFFEX_TS
Combining daily status df, exchange_product: CZCE_AP
Combining daily status df, exchange_product: CZCE_CF
Combining daily status df, exchange_product: CZCE_CJ
Combining daily status df, exchange_product: CZCE_CY
Combining daily status df, exchange_product: CZCE_FG
Combining daily status df, exchange_product: CZCE_JR
Combining daily status df, exchange_product: CZCE_LR
Combining daily status df, exchange_product: CZCE_MA
Combining daily status df, exchange_product: CZCE_ME
Combining daily status df, exchange_product: CZCE_OI
Combining daily status df, exchange_pro

## Trading Calendars

In [5]:
from pathlib import Path
import pandas as pd

fut_trade_cal_path = Path(r'D:\量化策略\CTA策略\Multi-factor Trend Following\db\rules_ingestion\data\raw\20260622\tushare\fut_trade_cal')

for exchange in ['CFFEX','CZCE', 'DCE', 'GFEX','INE','SHFE']:
    print(f'根据 Tushare - fut_trade_cal， 生成交易所交易日历：{exchange}')
    path = fut_trade_cal_path / f'exchange={exchange}.parquet'
    df = pd.read_parquet(path)
    # 仅保留 is_open=1 的日期
    trading_date_series = df.loc[df['is_open']==1,'cal_date']
    trading_date_series.rename('trading_date', inplace=True)
    
    out_path = Path(r'D:\量化策略\CTA策略\Multi-factor Trend Following\db\calendar\trading_calendars') / f'{exchange}.csv'
    trading_date_series.to_csv(out_path, index=False)



根据 Tushare - fut_trade_cal， 生成交易所交易日历：CFFEX
根据 Tushare - fut_trade_cal， 生成交易所交易日历：CZCE
根据 Tushare - fut_trade_cal， 生成交易所交易日历：DCE
根据 Tushare - fut_trade_cal， 生成交易所交易日历：GFEX
根据 Tushare - fut_trade_cal， 生成交易所交易日历：INE
根据 Tushare - fut_trade_cal， 生成交易所交易日历：SHFE
